In [51]:
import random
from pathlib import Path

In [52]:
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

In [53]:
RANDOM_SEED = 42
START       = 0
END         = 1

In [54]:
NUM_TRAIN_PAIRS = 8_000
NUM_VAL_PAIRS   = 1_000
NUM_TEST_PAIRS  = 1_000

In [55]:
DATASET_PATH = Path(
    '/group/pmc021/amunif/epi-thesis/workflow/'
    '16_Pairwise Ranking Healthy Liver/dataset/donor_3'
)

OUTPUT_PATH = Path(
    '/group/pmc021/amunif/epi-thesis/workflow/'
    '16_Pairwise Ranking Healthy Liver/output/baseline/donor_3'
)

In [56]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [57]:
# Load the data
data_df         = pl.read_parquet(DATASET_PATH / 'donor3_exp_histones.parquet')
permutation_lst = pl.read_parquet(
    DATASET_PATH / 'marker_combinations.parquet'
)["combination"].to_list()
 
print(f"Dataset shape : {data_df.shape}")
print(f"Combinations  : {len(permutation_lst)}")
data_df.head()

Dataset shape : (19645, 17)
Combinations  : 31


gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",17,100,"[0.0, 0.0, … 0.0]",23,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",33,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",37,100,"[0.0, 0.0, … 0.0]",42,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942


In [58]:
# Split train/val/test
all_idx = np.arange(len(data_df))
 
train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.20, random_state=RANDOM_SEED, shuffle=True
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=RANDOM_SEED, shuffle=True
)

In [59]:
assert not set(train_idx) & set(val_idx),  "Train/Val overlap!"
assert not set(train_idx) & set(test_idx), "Train/Test overlap!"
assert not set(val_idx)   & set(test_idx), "Val/Test overlap!"
 
print(f"Train : {len(train_idx):>6}  ({len(train_idx)/len(data_df)*100:.1f}%)")
print(f"Val   : {len(val_idx):>6}  ({len(val_idx)/len(data_df)*100:.1f}%)")
print(f"Test  : {len(test_idx):>6}  ({len(test_idx)/len(data_df)*100:.1f}%)")

Train :  15716  (80.0%)
Val   :   1964  (10.0%)
Test  :   1965  (10.0%)


In [60]:
# Function to generate gene pair
def generate_pairs(data_np, indexes, num_pairs, seed):
    """
    Returns (X, y):
      X : (num_pairs, 2 * feature_dim) — concatenated [gene1_feats | gene2_feats]
      y : (num_pairs,) int             — 1 if gene1_expr > gene2_expr, else 0
    """
    rng  = np.random.default_rng(seed)
    idx  = rng.choice(indexes, size=(num_pairs, 2), replace=True)
 
    feats1 = np.stack(data_np[idx[:, 0], 1])
    feats2 = np.stack(data_np[idx[:, 1], 1])
    val1   = data_np[idx[:, 0], 2].astype(float)
    val2   = data_np[idx[:, 1], 2].astype(float)
 
    X = np.concatenate([feats1, feats2], axis=1)
    y = (val1 > val2).astype(int)
    return X, y

In [61]:
# Function to calculate the label distribution
def label_distribution(y, split_name):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    print(f"  {split_name} label distribution:")
    for lbl, cnt in zip(unique, counts):
        print(f"    Label {int(lbl)}: {cnt:>6}  ({cnt/total*100:.1f}%)")

In [62]:
# Calculate the antisymmetric score
def antisymmetry_score(clf, X_test):
    """
    Swap the two feature halves to create (gene2, gene1) pairs and check
    that predictions flip. Returns fraction of pairs where pred(A,B) != pred(B,A).
    Perfectly antisymmetric = 1.0, random = ~0.5.
    """
    half      = X_test.shape[1] // 2
    X_swapped = np.concatenate([X_test[:, half:], X_test[:, :half]], axis=1)
 
    pred_orig    = clf.predict(X_test)
    pred_swapped = clf.predict(X_swapped)
 
    return float(np.mean(pred_orig + pred_swapped == 1))

In [63]:
# Build the classifier
def build_classifiers(random_state):
    return {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000, C=1.0, solver="lbfgs",
                random_state=random_state, n_jobs=-1,
            )),
        ]),
        "RandomForest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(
                n_estimators=200, max_depth=None, min_samples_leaf=4,
                random_state=random_state, n_jobs=-1,
            )),
        ]),
        "SVM_Linear": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(C=1.0, max_iter=2000, random_state=random_state, dual="auto")),
        ]),
    }

In [64]:
# Evaluate function
def evaluate(clf, X_test, y_test, clf_name, item_name):
    y_pred  = clf.predict(X_test)
    y_score = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") \
              else y_pred.astype(float)
 
    acc            = accuracy_score(y_test, y_pred) * 100
    auc            = roc_auc_score(y_test, y_score)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary"
    )
    antisym = antisymmetry_score(clf, X_test)
    cr      = classification_report(y_test, y_pred)
    cm      = confusion_matrix(y_test, y_pred)
 
    print(f"\n{'='*60}")
    print(f"  {clf_name}  |  {item_name}")
    print(f"{'='*60}")
    print(f"  Accuracy:           {acc:.2f}%")
    print(f"  AUC:                {auc:.4f}")
    print(f"  Precision:          {prec:.4f}")
    print(f"  Recall:             {rec:.4f}")
    print(f"  F1-Score:           {f1:.4f}")
    print(f"  Antisymmetry score: {antisym:.4f}  (1.0 = perfect, ~0.5 = random)")
    print(f"\n{cr}")
    print(f"Confusion Matrix:\n{cm}")
 
    return {
        "model":          clf_name,
        "seed":           RANDOM_SEED,
        "histone_marker": item_name,
        "test_accuracy":       round(acc,    4),
        "test_auc":            round(auc,    4),
        "test_precision":      round(prec,   4),
        "test_recall":         round(rec,    4),
        "test_f1":             round(f1,     4),
        "antisymmetry":   round(antisym,4),
    }

In [65]:
# Run the baseline
all_results = []
fitted_classifiers = {}

for number in range(START, END):
    item      = permutation_lst[number]
    item_name = '-'.join(item)
    print(f"\n>>> {number:03d}-{item_name}")
 
    # Build feature matrix for this histone combination
    df   = data_df.with_columns(histone=pl.concat_list(item))
    X_np = df["gene_id", "histone", "value"].to_numpy()
 
    # Generate pairs
    X_train, y_train = generate_pairs(X_np, train_idx, NUM_TRAIN_PAIRS, RANDOM_SEED)
    X_val,   y_val   = generate_pairs(X_np, val_idx,   NUM_VAL_PAIRS,   RANDOM_SEED + 1)
    X_test,  y_test  = generate_pairs(X_np, test_idx,  NUM_TEST_PAIRS,  RANDOM_SEED + 2)
 
    label_distribution(y_train, "Train")
    label_distribution(y_test,  "Test")
 
    classifiers = build_classifiers(RANDOM_SEED)
 
    for clf_name, clf in classifiers.items():
        print(f"\n  Training {clf_name}...")
        clf.fit(X_train, y_train)
 
        # Validation metrics
        y_val_pred = clf.predict(X_val)
        y_val_score = (clf.predict_proba(X_val)[:, 1]
                       if hasattr(clf, "predict_proba")
                       else clf.decision_function(X_val))
 
        val_acc  = accuracy_score(y_val, y_val_pred) * 100
        val_auc  = roc_auc_score(y_val, y_val_score)
        val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(
            y_val, y_val_pred, average="binary"
        )
 
        print(f"  [VAL] Accuracy: {val_acc:.2f}%  AUC: {val_auc:.4f}  "
              f"Precision: {val_prec:.4f}  Recall: {val_rec:.4f}  "
              f"F1: {val_f1:.4f}")
 
        # Test metrics
        result = evaluate(clf, X_test, y_test, clf_name, item_name)

        # Add the validation results
        result.update({
            "val_accuracy":  round(val_acc,  4),
            "val_auc":       round(val_auc,  4),
            "val_precision": round(val_prec, 4),
            "val_recall":    round(val_rec,  4),
            "val_f1":        round(val_f1,   4),
        })

        # Save the results
        all_results.append(result)
 
        # Save per-model CSV
        out_file = (OUTPUT_PATH /
                    f"{number:03d}-{item_name}-{clf_name}-seed{RANDOM_SEED}"
                    f"-test-metrics.csv")
        pl.DataFrame(result).write_csv(out_file)

    fitted_classifiers[item_name] = dict(classifiers)


>>> 000-H3K4me3
  Train label distribution:
    Label 0:   4111  (51.4%)
    Label 1:   3889  (48.6%)
  Test label distribution:
    Label 0:    509  (50.9%)
    Label 1:    491  (49.1%)

  Training LogisticRegression...


/group/pmc021/amunif/env/pytorch2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  [VAL] Accuracy: 71.90%  AUC: 0.7778  Precision: 0.7692  Recall: 0.6551  F1: 0.7076

  LogisticRegression  |  H3K4me3
  Accuracy:           71.90%
  AUC:                0.7958
  Precision:          0.7344
  Recall:             0.6701
  F1-Score:           0.7007
  Antisymmetry score: 0.7760  (1.0 = perfect, ~0.5 = random)

              precision    recall  f1-score   support

           0       0.71      0.77      0.74       509
           1       0.73      0.67      0.70       491

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000

Confusion Matrix:
[[390 119]
 [162 329]]

  Training RandomForest...
  [VAL] Accuracy: 73.40%  AUC: 0.8194  Precision: 0.7843  Recall: 0.6724  F1: 0.7241

  RandomForest  |  H3K4me3
  Accuracy:           73.70%
  AUC:                0.8146
  Precision:          0.7579
  Recall:             0.6823
  F1-Score:           0.7181
  Antisymmetry score

In [66]:
# Save the output
summary_df = pl.DataFrame(all_results)

column_order = [
    "model", "seed", "histone_marker",
    "val_accuracy", "val_auc", "val_precision", "val_recall", "val_f1",
    "test_accuracy", "test_auc", "test_precision", "test_recall", "test_f1",
    "antisymmetry",
]
summary_df = summary_df.select(column_order)

summary_df.write_csv(OUTPUT_PATH / f"baseline_summary_seed{RANDOM_SEED}.csv")
summary_df

model,seed,histone_marker,val_accuracy,val_auc,val_precision,val_recall,val_f1,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""LogisticRegression""",42,"""H3K4me3""",71.9,0.7778,0.7692,0.6551,0.7076,71.9,0.7958,0.7344,0.6701,0.7007,0.776
"""RandomForest""",42,"""H3K4me3""",73.4,0.8194,0.7843,0.6724,0.7241,73.7,0.8146,0.7579,0.6823,0.7181,0.806
"""SVM_Linear""",42,"""H3K4me3""",72.2,0.7794,0.7732,0.657,0.7104,71.7,0.7161,0.7321,0.668,0.6986,0.771
